# Notebook 01 — ETL: Raw TMDB CSV → Cleaned Parquet + Neo4j CSVs

**Dataset:** TMDB All Movies (Kaggle: alanvourch/tmdb-movies-daily-updates)  
**Business Question:** *Who are the highest-impact film industry professionals — actors, directors, producers, cinematographers, writers, and composers — and how do their collaboration networks drive box-office success?*

**Pipeline:**
```
TMDB_all_movies.csv
  → DuckDB (clean, filter, derive features)
  → clean_movies.parquet          (canonical snapshot)
  → Neo4j-ready CSVs              (nodes + relationships)
```

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")
RAW_CSV  = DATA_DIR / "TMDB_all_movies.csv"

assert RAW_CSV.exists(), f"Download TMDB_all_movies.csv from Kaggle and place at {RAW_CSV}"

con = duckdb.connect()
print("DuckDB version:", duckdb.__version__)

DuckDB version: 0.10.3


## Step 1 — Ingest Raw CSV into DuckDB

In [2]:
con.execute(f"""
    CREATE OR REPLACE TABLE raw_movies AS
    SELECT * FROM read_csv_auto('{RAW_CSV}',
        nullstr=['', 'NA', 'N/A', 'None'],
        header=true,
        ignore_errors=true
    )
""")

shape = con.execute("SELECT COUNT(*) AS rows, COUNT(COLUMNS(*)) AS cols FROM raw_movies").fetchdf()
print("Raw shape:", shape)
con.execute("DESCRIBE raw_movies").df()

Raw shape:       rows     cols   cols_1   cols_2   cols_3   cols_4   cols_5   cols_6  \
0  1184452  1184452  1184443  1184452  1184452  1184452  1060285  1184452   

    cols_7   cols_8  ...  cols_18  cols_19  cols_20  cols_21  cols_22  \
0  1184452  1184452  ...   753025   808269   993481   318303   599310   

   cols_23  cols_24  cols_25  cols_26  cols_27  
0   404304   134255   466891   466891   894581  

[1 rows x 29 columns]


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,title,VARCHAR,YES,None,None,None
2,vote_average,DOUBLE,YES,None,None,None
3,vote_count,DOUBLE,YES,None,None,None
4,status,VARCHAR,YES,None,None,None
5,release_date,DATE,YES,None,None,None
6,revenue,DOUBLE,YES,None,None,None
7,runtime,DOUBLE,YES,None,None,None
8,budget,DOUBLE,YES,None,None,None
9,imdb_id,VARCHAR,YES,None,None,None


## Step 2 — Cleaning & Feature Engineering

In [3]:
con.execute("""
    CREATE OR REPLACE TABLE clean_movies AS
    SELECT
        CAST(id AS BIGINT)                                      AS movie_id,
        TRIM(title)                                              AS title,
        TRIM(original_title)                                     AS original_title,
        TRY_CAST(release_date AS DATE)                           AS release_date,
        YEAR(TRY_CAST(release_date AS DATE))                     AS release_year,

        -- Crew fields (trimmed)
        TRIM(director)                                           AS director,
        TRIM(producers)                                          AS producers,
        TRIM(writers)                                            AS writers,
        TRIM(director_of_photography)                            AS dop,
        TRIM(music_composer)                                     AS composer,
        TRIM("cast")                                             AS cast_list,

        -- Content
        TRIM(genres)                                             AS genres,
        TRIM(overview)                                           AS overview,
        TRIM(tagline)                                            AS tagline,
        TRIM(original_language)                                  AS original_language,
        TRIM(production_companies)                               AS production_companies,

        -- Numerics: treat 0 as missing for financials
        COALESCE(TRY_CAST(popularity AS DOUBLE), 0.0)            AS popularity,
        COALESCE(TRY_CAST(vote_average AS DOUBLE), 0.0)          AS vote_average,
        COALESCE(TRY_CAST(vote_count AS BIGINT), 0)              AS vote_count,
        COALESCE(TRY_CAST(runtime AS INTEGER), 0)                AS runtime,
        NULLIF(TRY_CAST(budget AS DOUBLE), 0)                    AS budget,
        NULLIF(TRY_CAST(revenue AS DOUBLE), 0)                   AS revenue,

        -- Derived: ROI
        CASE
            WHEN NULLIF(TRY_CAST(budget AS DOUBLE), 0) IS NOT NULL
             AND NULLIF(TRY_CAST(revenue AS DOUBLE), 0) IS NOT NULL
            THEN ROUND(
                (NULLIF(TRY_CAST(revenue AS DOUBLE), 0) - NULLIF(TRY_CAST(budget AS DOUBLE), 0))
                / NULLIF(TRY_CAST(budget AS DOUBLE), 0) * 100.0
            , 2)
            ELSE NULL
        END                                                       AS roi_pct,

        -- Success label for ML (top-25% vote_average with sufficient votes)
        CASE
            WHEN TRY_CAST(vote_count AS BIGINT) >= 50
             AND TRY_CAST(vote_average AS DOUBLE) >= 7.0
            THEN 1 ELSE 0
        END                                                       AS is_successful

    FROM raw_movies
    WHERE id IS NOT NULL
      AND title IS NOT NULL
      AND TRIM(title) != ''
""")

count = con.execute("SELECT COUNT(*) FROM clean_movies").fetchone()[0]
print(f"Clean movies: {count:,}")
con.execute("SELECT * FROM clean_movies LIMIT 3").df()

Clean movies: 1,184,440


,movie_id,title,original_title,release_date,release_year,director,producers,writers,dop,composer,...,original_language,production_companies,popularity,vote_average,vote_count,runtime,budget,revenue,roi_pct,is_successful
0,2,Ariel,Ariel,1988-10-21,1988,Aki Kaurismäki,Aki Kaurismäki,Aki Kaurismäki,Timo Salminen,None,...,fi,Villealfa Filmproductions,1.6384,7.106,371,73,NaN,NaN,NaN,1
1,3,Shadows in Paradise,Varjoja paratiisissa,1986-10-17,1986,Aki Kaurismäki,Mika Kaurismäki,Aki Kaurismäki,Timo Salminen,None,...,fi,Villealfa Filmproductions,2.7136,7.300,435,74,NaN,NaN,NaN,1
2,5,Four Rooms,Four Rooms,1995-12-09,1995,"Robert Rodriguez, Allison Anders, Quentin Tara...","Quentin Tarantino, Alexandre Rockwell, Lawrenc...","Allison Anders, Robert Rodriguez, Alexandre Ro...","Rodrigo García, Guillermo Navarro, Phil Parmet...",Combustible Edison,...,en,"Miramax, A Band Apart",2.6464,5.900,2819,98,4000000.0,4257354.0,6.43,0


## Step 3 — Quality Audit

In [4]:
con.execute("""
    SELECT
        COUNT(*)                                           AS total,
        COUNT(director)                                    AS has_director,
        COUNT(producers)                                   AS has_producers,
        COUNT(dop)                                         AS has_dop,
        COUNT(composer)                                    AS has_composer,
        COUNT(writers)                                     AS has_writers,
        COUNT(cast_list)                                   AS has_cast,
        COUNT(budget)                                      AS has_budget,
        COUNT(revenue)                                     AS has_revenue,
        COUNT(roi_pct)                                     AS has_roi,
        SUM(is_successful)                                 AS successful_movies
    FROM clean_movies
""").df()

,total,has_director,has_producers,has_dop,has_composer,has_writers,has_cast,has_budget,has_revenue,has_roi,successful_movies
0,1184440,993475,404302,318301,134254,599307,808264,77836,27306,17641,9007.0


## Step 4 — Export Parquet Snapshot

In [5]:
PARQUET_PATH = DATA_DIR / "clean_movies.parquet"
con.execute(f"COPY clean_movies TO '{PARQUET_PATH}' (FORMAT PARQUET, COMPRESSION SNAPPY)")
import os
print(f"Parquet saved: {PARQUET_PATH}  ({os.path.getsize(PARQUET_PATH)/1e6:.1f} MB)")

Parquet saved: ../data/clean_movies.parquet  (405.2 MB)


## Step 5 — Export Neo4j Node CSVs

In [6]:
NEO_DIR = DATA_DIR / "neo4j_csv"
NEO_DIR.mkdir(parents=True, exist_ok=True)

# ── Movie nodes ──────────────────────────────────────────────────────────
con.execute(f"""
    COPY (
        SELECT DISTINCT
            movie_id, title, release_year, vote_average, vote_count,
            runtime, budget, revenue, roi_pct, original_language,
            genres, overview, tagline, popularity, is_successful
        FROM clean_movies
        WHERE movie_id IS NOT NULL AND title IS NOT NULL
    ) TO '{NEO_DIR}/nodes_Movie.csv' (HEADER, FORMAT CSV)
""")
n = con.execute(f"SELECT COUNT(*) FROM '{NEO_DIR}/nodes_Movie.csv'").fetchone()[0]
print(f"Movie nodes: {n:,}")

# ── Director nodes ───────────────────────────────────────────────────────
con.execute(f"""
    COPY (
        SELECT DISTINCT TRIM(director) AS name
        FROM clean_movies
        WHERE director IS NOT NULL AND TRIM(director) NOT IN ('', 'unknown')
    ) TO '{NEO_DIR}/nodes_Director.csv' (HEADER, FORMAT CSV)
""")
n = con.execute(f"SELECT COUNT(*) FROM '{NEO_DIR}/nodes_Director.csv'").fetchone()[0]
print(f"Director nodes: {n:,}")

# ── Actor nodes ──────────────────────────────────────────────────────────
con.execute(f"""
    COPY (
        SELECT DISTINCT TRIM(actor_name) AS name
        FROM (
            SELECT UNNEST(string_split(cast_list, ',')) AS actor_name
            FROM clean_movies
            WHERE cast_list IS NOT NULL AND TRIM(cast_list) != ''
        )
        WHERE TRIM(actor_name) NOT IN ('', 'unknown')
    ) TO '{NEO_DIR}/nodes_Actor.csv' (HEADER, FORMAT CSV)
""")
n = con.execute(f"SELECT COUNT(*) FROM '{NEO_DIR}/nodes_Actor.csv'").fetchone()[0]
print(f"Actor nodes: {n:,}")

# ── Producer nodes ───────────────────────────────────────────────────────
con.execute(f"""
    COPY (
        SELECT DISTINCT TRIM(p) AS name
        FROM (
            SELECT UNNEST(string_split(producers, ',')) AS p
            FROM clean_movies
            WHERE producers IS NOT NULL AND TRIM(producers) != ''
        )
        WHERE TRIM(p) NOT IN ('', 'unknown') AND LENGTH(TRIM(p)) > 1
    ) TO '{NEO_DIR}/nodes_Producer.csv' (HEADER, FORMAT CSV)
""")
n = con.execute(f"SELECT COUNT(*) FROM '{NEO_DIR}/nodes_Producer.csv'").fetchone()[0]
print(f"Producer nodes: {n:,}")

# ── Writer nodes ─────────────────────────────────────────────────────────
con.execute(f"""
    COPY (
        SELECT DISTINCT TRIM(w) AS name
        FROM (
            SELECT UNNEST(string_split(writers, ',')) AS w
            FROM clean_movies
            WHERE writers IS NOT NULL AND TRIM(writers) != ''
        )
        WHERE TRIM(w) NOT IN ('', 'unknown') AND LENGTH(TRIM(w)) > 1
    ) TO '{NEO_DIR}/nodes_Writer.csv' (HEADER, FORMAT CSV)
""")
n = con.execute(f"SELECT COUNT(*) FROM '{NEO_DIR}/nodes_Writer.csv'").fetchone()[0]
print(f"Writer nodes: {n:,}")

# ── DOP nodes ────────────────────────────────────────────────────────────
con.execute(f"""
    COPY (
        SELECT DISTINCT TRIM(dop) AS name
        FROM clean_movies
        WHERE dop IS NOT NULL AND TRIM(dop) NOT IN ('', 'unknown')
    ) TO '{NEO_DIR}/nodes_DOP.csv' (HEADER, FORMAT CSV)
""")
n = con.execute(f"SELECT COUNT(*) FROM '{NEO_DIR}/nodes_DOP.csv'").fetchone()[0]
print(f"DOP nodes: {n:,}")

# ── Composer nodes ───────────────────────────────────────────────────────
con.execute(f"""
    COPY (
        SELECT DISTINCT TRIM(composer) AS name
        FROM clean_movies
        WHERE composer IS NOT NULL AND TRIM(composer) NOT IN ('', 'unknown')
    ) TO '{NEO_DIR}/nodes_Composer.csv' (HEADER, FORMAT CSV)
""")
n = con.execute(f"SELECT COUNT(*) FROM '{NEO_DIR}/nodes_Composer.csv'").fetchone()[0]
print(f"Composer nodes: {n:,}")

Movie nodes: 1,184,440
Director nodes: 411,594
Actor nodes: 2,102,853
Producer nodes: 343,009
Writer nodes: 398,481
DOP nodes: 116,136
Composer nodes: 45,239


## Step 6 — Export Neo4j Relationship CSVs

In [7]:
# DIRECTED_BY
con.execute(f"""
    COPY (
        SELECT DISTINCT movie_id, TRIM(director) AS director_name
        FROM clean_movies
        WHERE movie_id IS NOT NULL AND director IS NOT NULL
          AND TRIM(director) NOT IN ('', 'unknown')
    ) TO '{NEO_DIR}/rel_DIRECTED_BY.csv' (HEADER, FORMAT CSV)
""")
n = con.execute(f"SELECT COUNT(*) FROM '{NEO_DIR}/rel_DIRECTED_BY.csv'").fetchone()[0]
print(f"DIRECTED_BY: {n:,}")

# ACTED_IN
con.execute(f"""
    COPY (
        SELECT DISTINCT movie_id, TRIM(actor_name) AS actor_name
        FROM (
            SELECT movie_id, UNNEST(string_split(cast_list, ',')) AS actor_name
            FROM clean_movies
            WHERE cast_list IS NOT NULL
        )
        WHERE TRIM(actor_name) NOT IN ('', 'unknown')
    ) TO '{NEO_DIR}/rel_ACTED_IN.csv' (HEADER, FORMAT CSV)
""")
n = con.execute(f"SELECT COUNT(*) FROM '{NEO_DIR}/rel_ACTED_IN.csv'").fetchone()[0]
print(f"ACTED_IN: {n:,}")

# PRODUCED_BY
con.execute(f"""
    COPY (
        SELECT DISTINCT movie_id, TRIM(p) AS producer_name
        FROM (
            SELECT movie_id, UNNEST(string_split(producers, ',')) AS p
            FROM clean_movies
            WHERE producers IS NOT NULL
        )
        WHERE TRIM(p) NOT IN ('', 'unknown') AND LENGTH(TRIM(p)) > 1
    ) TO '{NEO_DIR}/rel_PRODUCED_BY.csv' (HEADER, FORMAT CSV)
""")
n = con.execute(f"SELECT COUNT(*) FROM '{NEO_DIR}/rel_PRODUCED_BY.csv'").fetchone()[0]
print(f"PRODUCED_BY: {n:,}")

# WRITTEN_BY
con.execute(f"""
    COPY (
        SELECT DISTINCT movie_id, TRIM(w) AS writer_name
        FROM (
            SELECT movie_id, UNNEST(string_split(writers, ',')) AS w
            FROM clean_movies
            WHERE writers IS NOT NULL
        )
        WHERE TRIM(w) NOT IN ('', 'unknown') AND LENGTH(TRIM(w)) > 1
    ) TO '{NEO_DIR}/rel_WRITTEN_BY.csv' (HEADER, FORMAT CSV)
""")
n = con.execute(f"SELECT COUNT(*) FROM '{NEO_DIR}/rel_WRITTEN_BY.csv'").fetchone()[0]
print(f"WRITTEN_BY: {n:,}")

# SHOT_BY
con.execute(f"""
    COPY (
        SELECT DISTINCT movie_id, TRIM(dop) AS dop_name
        FROM clean_movies
        WHERE movie_id IS NOT NULL AND dop IS NOT NULL
          AND TRIM(dop) NOT IN ('', 'unknown')
    ) TO '{NEO_DIR}/rel_SHOT_BY.csv' (HEADER, FORMAT CSV)
""")
n = con.execute(f"SELECT COUNT(*) FROM '{NEO_DIR}/rel_SHOT_BY.csv'").fetchone()[0]
print(f"SHOT_BY: {n:,}")

# SCORE_BY
con.execute(f"""
    COPY (
        SELECT DISTINCT movie_id, TRIM(composer) AS composer_name
        FROM clean_movies
        WHERE movie_id IS NOT NULL AND composer IS NOT NULL
          AND TRIM(composer) NOT IN ('', 'unknown')
    ) TO '{NEO_DIR}/rel_SCORE_BY.csv' (HEADER, FORMAT CSV)
""")
n = con.execute(f"SELECT COUNT(*) FROM '{NEO_DIR}/rel_SCORE_BY.csv'").fetchone()[0]
print(f"SCORE_BY: {n:,}")

print("\n✅ All Neo4j CSVs exported to", NEO_DIR)

DIRECTED_BY: 993,475
ACTED_IN: 7,386,327
PRODUCED_BY: 867,147
WRITTEN_BY: 973,028
SHOT_BY: 318,301
SCORE_BY: 134,254

✅ All Neo4j CSVs exported to ../data/neo4j_csv


## Step 7 — Verify with DuckDB Analytics

In [8]:
# Top directors by avg ROI (business preview)
con.execute("""
    SELECT director,
           COUNT(*) AS films,
           ROUND(AVG(roi_pct),1) AS avg_roi,
           ROUND(AVG(vote_average),2) AS avg_rating
    FROM clean_movies
    WHERE director IS NOT NULL
      AND roi_pct IS NOT NULL
    GROUP BY director
    HAVING COUNT(*) >= 3
    ORDER BY avg_roi DESC
    LIMIT 10
""").df()

,director,films,avg_roi,avg_rating
0,"Deu, Herman Gater",3,99999900.0,0.00
1,Ron Howard,26,9449284.2,6.72
2,Matthew Cavanaugh,7,4801332.4,8.57
3,Maria Pereira,4,2502425.0,5.00
4,Kimo Stamboel,3,299956.5,6.10
5,P. Vasu,3,116603.4,3.17
6,Ram Babu Gurung,4,69583.7,6.93
7,Alexis Moreno Muñóz,4,31900.0,0.00
8,"Brennen Rowley, Noah Geertsen",4,26650.0,0.00
9,Roman Karimov,4,25446.6,5.90


In [9]:
# Top actors by number of successful films
con.execute("""
    SELECT TRIM(actor_name) AS actor,
           COUNT(*) AS total_films,
           SUM(is_successful) AS successful_films,
           ROUND(AVG(vote_average),2) AS avg_rating
    FROM (
        SELECT UNNEST(string_split(cast_list, ',')) AS actor_name,
               is_successful, vote_average
        FROM clean_movies
        WHERE cast_list IS NOT NULL
    )
    WHERE TRIM(actor_name) NOT IN ('', 'unknown')
    GROUP BY TRIM(actor_name)
    HAVING COUNT(*) >= 5
    ORDER BY successful_films DESC
    LIMIT 10
""").df()

,actor,total_films,successful_films,avg_rating
0,Frank Welker,520,129.0,6.44
1,Grey DeLisle,246,85.0,6.44
2,Bess Flowers,741,75.0,5.63
3,Jr.,1289,69.0,4.29
4,Sam Harris,320,63.0,6.03
5,Fred Tatasciore,195,57.0,6.05
6,Tara Strong,247,53.0,5.85
7,Bert Stevens,208,51.0,6.28
8,Dee Bradley Baker,263,50.0,5.63
9,Koichi Yamadera,282,49.0,5.41
